# Chapter 0 — Setup

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/00_setup.ipynb)

This notebook installs [AGAMA](https://github.com/GalacticDynamics-Oxford/Agama) and checks that it reproduces
analytic results for a Plummer sphere. Units: kpc, km/s, M$_\odot$ (time unit ≈ 0.978 Gyr).

In [ ]:
# Install AGAMA if missing. On Colab (Linux x86_64) this uses a ready-made wheel (seconds);
# elsewhere, follow https://github.com/GalacticDynamics-Oxford/Agama/blob/master/INSTALL
import sys, importlib.util
if importlib.util.find_spec('agama') is None:
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama
print(agama.__version__)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

agama.setUnits(mass=1, length=1, velocity=1)   # Msun, kpc, km/s
M, b = 1e10, 1.0
pot = agama.Potential(type='Plummer', mass=M, scaleRadius=b)

# circular speed: AGAMA vs analytic v_c^2 = G M r^2 / (r^2 + b^2)^{3/2}
r = np.logspace(-2, 2, 200)
vc_agama = np.sqrt(-r * pot.force(np.column_stack([r, 0*r, 0*r]))[:, 0])
vc_exact = np.sqrt(agama.G * M * r**2 / (r**2 + b**2)**1.5)

plt.loglog(r, vc_exact, 'k-', lw=3, alpha=0.3, label='analytic')
plt.loglog(r, vc_agama, 'r--', label='AGAMA')
plt.axvline(np.sqrt(2) * b, color='gray', ls=':', label=r'peak at $\sqrt{2}\,b$')
plt.xlabel('r [kpc]'); plt.ylabel(r'$v_c$ [km/s]'); plt.legend(); plt.show()

err = np.max(np.abs(vc_agama / vc_exact - 1))
print(f'max relative error in v_c: {err:.1e}')
assert err < 1e-10

In [ ]:
# orbit: energy conservation and constancy of the radial action
_, traj = agama.orbit(potential=pot, ic=[1, 0, 0, 0, 150, 30], time=10.0, trajsize=2001)
E = pot.potential(traj[:, :3]) + 0.5 * np.sum(traj[:, 3:]**2, axis=1)
Jr = agama.ActionFinder(pot)(traj[::200])[:, 0]
print(f'max |dE/E| = {np.max(np.abs(E/E[0]-1)):.1e},  std(Jr)/mean(Jr) = {np.std(Jr)/np.mean(Jr):.1e}')
assert np.max(np.abs(E/E[0]-1)) < 1e-5 and np.std(Jr)/np.mean(Jr) < 1e-4
print('AGAMA works.')